# Lab 2B: ML system design: model selection, bias-variance, business metrics and trade-offs
**Presenter: Nagarjuna Naidu**

**Learning outcomes**
- Choose a model with cross-validation and a sane baseline ("start simple, earn complexity"), and read bias-variance from curves.
- Turn a model score into money: cost matrix, optimal threshold, and the accuracy / latency / cost trade-off.
- Reason about drift, retraining, versioning and weighted KPIs when choosing an AI design.

**Time budget:** ~30 min core (sections 1-9). Case: *predict which loan applications will default* (synthetic, offline, sklearn only, runtime about 1 min).

## 0. Environment check

In [ ]:
import os
# single-threaded math libs: reproducible results and no thread hangs on mixed laptops
for v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[v] = "1"
import sys, platform, warnings, time, json, hashlib, datetime, pathlib
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, sklearn, scipy, joblib, matplotlib
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)
INK, TEAL, LIME, CORAL, GOLD, PURPLE = "#0B1F2A", "#0E7C86", "#9BD13B", "#E4572E", "#F2B134", "#6C4AB6"
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[TEAL, CORAL, GOLD, PURPLE, LIME, INK]),
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": INK,
    "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "axes.grid": True, "grid.alpha": 0.25, "figure.dpi": 100, "axes.titleweight": "bold",
})
pathlib.Path("artifacts").mkdir(exist_ok=True)
print("python", sys.version.split()[0], "|", platform.system(),
      "| numpy", np.__version__, "| pandas", pd.__version__, "| sklearn", sklearn.__version__)

## 1. Setup + data
- One synthetic business case, fixed seed: ~6,000 loan applications, ~12% defaults (class imbalance).
- 11 columns: some informative, one categorical (`region`), a few pure noise.
- Split: stratified train 60% / valid 20% / test 20%. Test stays untouched until the end.
- Real life: **split by time** (train on old, test on new) to avoid leakage. We do that in section 6 (drift).

In [ ]:
def make_loans(n, seed, drift=0.0):
    """Synthetic loan book. `drift` (0..1) changes how the world works (used in section 6)."""
    r = np.random.default_rng(seed)
    income = r.lognormal(10.9, 0.45, n)
    credit = np.clip(r.normal(680, 60, n), 300, 850)
    dti = np.clip(r.beta(3, 6, n), 0, 1)
    util = r.beta(2, 3, n)
    loan = income * r.uniform(0.1, 0.8, n)
    term = r.choice([36, 60, 84], n, p=[0.4, 0.4, 0.2])
    emp = np.clip(r.exponential(5, n), 0, 40)
    delinq = r.poisson(0.4, n)
    region = r.choice(list("ABCD"), n)
    age = np.clip(r.normal(40, 12, n), 18, 80)
    noise_a, noise_b = r.normal(size=n), r.normal(size=n)
    z = lambda x, m, s: (x - m) / s
    lti = loan / income
    logit = (-3.4 - 0.7 * (1 - 0.7 * drift) * z(credit, 680, 60)
             + 0.6 * z(dti, 0.33, 0.15) + 0.5 * (1 + drift) * delinq
             + 0.8 * (1 + 2.5 * drift) * z(util, 0.4, 0.2)
             + 1.6 * (z(dti, 0.33, 0.15) > 1) * (z(util, 0.4, 0.2) > 0.5)   # interaction (non-linear)
             + 0.4 * (z(age, 40, 12) ** 2 - 1)                            # U-shaped age effect (non-linear)
             + 0.5 * (term == 84) + 0.9 * z(lti, 0.45, 0.2) - 0.2 * z(emp, 5, 5)
             + pd.Series(region).map({"A": 0, "B": 0.2, "C": -0.2, "D": 0.4}).values)
    y = (r.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
    df = pd.DataFrame(dict(income=income.round(0), credit_score=credit.round(0), debt_to_income=dti.round(3),
                           utilisation=util.round(3), loan_amount=loan.round(0), term_months=term,
                           employment_years=emp.round(1), num_delinquencies=delinq, age=age.round(0),
                           noise_a=noise_a.round(3), noise_b=noise_b.round(3), region=region))
    return df, pd.Series(y, name="default")

from sklearn.model_selection import train_test_split
X, y = make_loans(6000, SEED)
NUM = [c for c in X.columns if c != "region"]; CAT = ["region"]
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
X_train, X_valid, y_train, y_valid = train_test_split(X_tmp, y_tmp, test_size=0.25, stratify=y_tmp, random_state=SEED)
print(f"rows={len(X)}  default rate={y.mean():.1%}  train/valid/test = {len(X_train)}/{len(X_valid)}/{len(X_test)}")
X_train.head(3)

## 2. Model selection: baselines first, then earn complexity
- Baselines: majority class, and a hand-written risk rule a credit officer might use.
- Candidates: LogisticRegression, RandomForest, HistGradientBoosting.
- 5-fold stratified CV on the **train** set: mean +/- std of ROC-AUC and PR-AUC (PR-AUC of a random model = default rate, about 0.12).

In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import roc_auc_score, average_precision_score

class RuleBaseline(ClassifierMixin, BaseEstimator):
    """Hand-written rule: risky if low credit score and/or high debt-to-income."""
    def fit(self, X, y): self.classes_ = np.array([0, 1]); return self
    def predict_proba(self, X):
        s = 0.5 * np.clip(X["debt_to_income"] / 0.5, 0, 1) + 0.5 * np.clip((720 - X["credit_score"]) / 180, 0, 1)
        s = s.to_numpy(); return np.c_[1 - s, s]
    def predict(self, X): return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

def build(est):
    pre = ColumnTransformer([("num", StandardScaler(), NUM), ("cat", OneHotEncoder(handle_unknown="ignore"), CAT)])
    return Pipeline([("prep", pre), ("model", est)])

MODELS = {
    "Majority class": DummyClassifier(strategy="prior"),
    "Simple rule": RuleBaseline(),
    "LogisticRegression": build(LogisticRegression(max_iter=1000)),
    "RandomForest": build(RandomForestClassifier(n_estimators=150, min_samples_leaf=5, random_state=SEED, n_jobs=1)),
    "HistGradientBoosting": build(HistGradientBoostingClassifier(max_iter=150, learning_rate=0.05, max_depth=4, random_state=SEED)),
}
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
rows = []
for name, m in MODELS.items():
    r = cross_validate(m, X_train, y_train, cv=cv, scoring={"auc": "roc_auc", "pr": "average_precision"}, n_jobs=1)
    rows.append({"model": name, "CV AUC": f"{r['test_auc'].mean():.3f} +/- {r['test_auc'].std():.3f}",
                 "CV PR-AUC": f"{r['test_pr'].mean():.3f} +/- {r['test_pr'].std():.3f}", "_auc": r["test_auc"].mean()})
res = pd.DataFrame(rows)
real = res[res.model.isin(["LogisticRegression", "RandomForest", "HistGradientBoosting"])]
best_name = real.loc[real._auc.idxmax(), "model"]
print(res.drop(columns="_auc").to_string(index=False))
print("\nBest by CV AUC:", best_name)

**Takeaway: start simple, earn complexity.**
- Look at the gap between the rule/logistic baseline and the heavier models: is it worth the extra latency, size and maintenance (section 5)?
- If two models are within one std of each other, prefer the simpler one.

## 3. Bias-variance
- **Underfit (high bias):** train and validation scores both low. **Overfit (high variance):** train high, validation lower, big gap.
- Validation curve: vary model complexity (tree `max_depth`). Learning curve: vary the amount of training data.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import validation_curve
depths = np.arange(1, 17)
tr, va = validation_curve(build(DecisionTreeClassifier(random_state=SEED)), X_train, y_train,
                          param_name="model__max_depth", param_range=depths, cv=cv, scoring="roc_auc", n_jobs=1)
tr_m, va_m = tr.mean(1), va.mean(1)
best_d = int(depths[va_m.argmax()])
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(depths, tr_m, "-o", color=CORAL, label="train AUC"); ax.plot(depths, va_m, "-o", color=TEAL, label="validation AUC")
ax.axvspan(0.5, 2.5, color=GOLD, alpha=0.15); ax.axvspan(best_d - 1, best_d + 1, color=LIME, alpha=0.2); ax.axvspan(10.5, 16.5, color=CORAL, alpha=0.1)
top = ax.get_ylim()[1]
ax.text(1.5, top, "underfit", ha="center", va="top"); ax.text(best_d, top, "sweet spot", ha="center", va="top"); ax.text(13.5, top, "overfit", ha="center", va="top")
ax.set(xlabel="tree max_depth (model complexity)", ylabel="ROC-AUC", title="Validation curve: complexity vs score"); ax.legend(loc="center right")
plt.tight_layout(); plt.show()
print(f"best depth={best_d}: val AUC={va_m.max():.3f} | depth 16: train={tr_m[-1]:.3f} val={va_m[-1]:.3f} (gap {tr_m[-1]-va_m[-1]:.3f})")

In [ ]:
from sklearn.model_selection import learning_curve
cases = {"Underfit: depth=1": dict(max_depth=1), f"Sweet spot: depth={best_d}": dict(max_depth=best_d), "Overfit: depth=None": dict(max_depth=None)}
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True)
for ax, (title, p) in zip(axes, cases.items()):
    n, trs, vas = learning_curve(build(DecisionTreeClassifier(random_state=SEED, **p)), X_train, y_train, cv=cv,
                                 train_sizes=np.linspace(0.1, 1, 6), scoring="roc_auc", n_jobs=1)
    ax.plot(n, trs.mean(1), "-o", color=CORAL, label="train"); ax.plot(n, vas.mean(1), "-o", color=TEAL, label="validation")
    ax.set(title=title, xlabel="training rows"); ax.set_ylim(0.45, 1.02)
    print(f"{title:<24} final train={trs.mean(1)[-1]:.3f}  val={vas.mean(1)[-1]:.3f}  gap={trs.mean(1)[-1]-vas.mean(1)[-1]:.3f}")
axes[0].set_ylabel("ROC-AUC"); axes[0].legend(); plt.tight_layout(); plt.show()

Reading the curves:
- **Underfit:** curves meet at a low score. More data will not help; need a more expressive model or better features.
- **Overfit:** big gap that narrows slowly. More data, regularisation or a simpler model helps.

### Your turn 1: tame the overfit tree
Change the depth and/or add regularisation (`min_samples_leaf`), then judge from the train-vs-validation **gap**.
- Goal: validation AUC close to the sweet spot AND a small gap.

In [ ]:
def gap_report(params):
    m = build(DecisionTreeClassifier(random_state=SEED, **params))
    r = cross_validate(m, X_train, y_train, cv=cv, scoring="roc_auc", return_train_score=True, n_jobs=1)
    t, v = r["train_score"].mean(), r["test_score"].mean()
    print(f"{params}: train={t:.3f}  val={v:.3f}  gap={t-v:.3f}")
    return t, v

gap_report(dict(max_depth=None))              # the overfit reference
my_params = dict(max_depth=None)              # TODO: try max_depth=6, then add min_samples_leaf=30 (hint: leaf size is regularisation)
gap_report(my_params)

## 4. Business metrics: from AUC to money
- A false negative (missed default) and a false positive (rejected good customer) cost very different amounts.
- **Cost matrix (parameters given, illustrative):** missed default = **$5,000** (loss given default); rejected good customer = **$500** (lost margin); correct decisions = $0 (simplification).
- We pick the decision threshold on **validation** by minimising expected cost, then confirm on **test**.

In [ ]:
C_FN, C_FP = 5000, 500
def total_cost(y_true, flag, c_fn=C_FN, c_fp=C_FP):
    y_true, flag = np.asarray(y_true), np.asarray(flag)
    return ((y_true == 1) & (flag == 0)).sum() * c_fn + ((y_true == 0) & (flag == 1)).sum() * c_fp

def best_threshold(p, y_true, c_fn=C_FN, c_fp=C_FP, grid=np.linspace(0.01, 0.99, 99)):
    costs = np.array([total_cost(y_true, p >= t, c_fn, c_fp) for t in grid])
    return grid[costs.argmin()], costs

best_model = clone(MODELS[best_name]).fit(X_train, y_train)
p_valid = best_model.predict_proba(X_valid)[:, 1]; p_test = best_model.predict_proba(X_test)[:, 1]
grid = np.linspace(0.01, 0.99, 99)
t_star, costs_valid = best_threshold(p_valid, y_valid)
per1k = 1000 / len(y_valid)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(grid, costs_valid * per1k / 1000, color=TEAL, lw=2)
ax.axvline(0.5, color=CORAL, ls="--", label="default 0.5"); ax.axvline(t_star, color=LIME, lw=3, label=f"cost-optimal {t_star:.2f}")
ax.set(xlabel="decision threshold (flag as default if P >= t)", ylabel="expected cost per 1,000 applications ($ thousand)",
       title=f"Expected cost vs threshold ({best_name}, validation)"); ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
rule = MODELS["Simple rule"].fit(X_train, y_train)
n_test = len(y_test)
scen = {
    "Approve everyone (no model)": np.zeros(n_test, int),
    "Simple rule": rule.predict(X_test),
    f"{best_name} @ 0.5": (p_test >= 0.5).astype(int),
    f"{best_name} @ {t_star:.2f} (cost-optimal)": (p_test >= t_star).astype(int),
}
out = pd.DataFrame({k: {"cost per 1,000 apps ($)": total_cost(y_test, f) * 1000 / n_test, "flag rate": f.mean()} for k, f in scen.items()}).T
print(out.round(3).to_string())
c_rule, c_opt = out.iloc[1, 0], out.iloc[3, 0]
print(f"\nOptimal threshold {t_star:.2f} vs 0.5. Uplift vs rule: ${c_rule - c_opt:,.0f} per 1,000 applications "
      f"= ${(c_rule - c_opt) * 100:,.0f} per 100,000 applications (test set, illustrative cost matrix).")

### Your turn 2: what if a missed default costs more?
Change the cost of a missed default (`NEW_C_FN`) and watch the optimal threshold move.
- Predict first: higher or lower threshold? Why?

In [ ]:
NEW_C_FN = 5000          # TODO: try 20000 (e.g. loan is larger or collateral is weak)
t_new, _ = best_threshold(p_valid, y_valid, c_fn=NEW_C_FN)
print(f"C_FN={NEW_C_FN}: optimal threshold = {t_new:.2f}  (baseline at C_FN=5000: {t_star:.2f})")

## 5. Training vs inference: accuracy vs latency vs cost
- Training is paid once (or per retrain); inference is paid on **every request**, so it dominates cost at scale.
- We measure: fit time, single-request latency (p50/p95), 1000-row batch time, size on disk.
- Cost per 1M requests uses an **illustrative assumption**: one compute instance at **$0.20 per hour** (not a real quote), handling one request at a time at full utilisation.

In [ ]:
PRICE_PER_HOUR = 0.20   # ILLUSTRATIVE compute price assumption (USD/hour)
tmp = pathlib.Path("artifacts/_size_check"); tmp.mkdir(exist_ok=True)
cand = ["LogisticRegression", "RandomForest", "HistGradientBoosting"]
perf = []
for name in cand:
    m = clone(MODELS[name]); t0 = time.perf_counter(); m.fit(X_train, y_train); fit_s = time.perf_counter() - t0
    m.predict_proba(X_test.iloc[[0]])                                  # warm-up
    lat = []
    for i in range(150):
        row = X_test.iloc[[i]]; t0 = time.perf_counter(); m.predict_proba(row); lat.append((time.perf_counter() - t0) * 1000)
    t0 = time.perf_counter(); pb = m.predict_proba(X_test.iloc[:1000] if len(X_test) >= 1000 else X_test); batch_ms = (time.perf_counter() - t0) * 1000
    joblib.dump(m, tmp / f"{name}.joblib"); size_kb = (tmp / f"{name}.joblib").stat().st_size / 1024
    cost_online = 1e6 * (np.mean(lat) / 1000) / 3600 * PRICE_PER_HOUR
    cost_batch = (1e6 / len(pb)) * (batch_ms / 1000) / 3600 * PRICE_PER_HOUR
    perf.append(dict(model=name, test_AUC=roc_auc_score(y_test, m.predict_proba(X_test)[:, 1]), fit_s=fit_s,
                     p50_ms=np.percentile(lat, 50), p95_ms=np.percentile(lat, 95), batch_1000_ms=batch_ms,
                     size_KB=size_kb, cost_1M_online_usd=cost_online, cost_1M_batch_usd=cost_batch))
perf = pd.DataFrame(perf)
print(perf.round(3).to_string(index=False))
print(f"\nILLUSTRATIVE: cost per 1M requests = requests x latency x (${PRICE_PER_HOUR}/hour); real prices, autoscaling and idle time differ.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
cols = [TEAL, CORAL, PURPLE]
for (_, r), c in zip(perf.iterrows(), cols):
    ax.scatter(r.p95_ms, r.test_AUC, s=300 + 6000 * r.cost_1M_online_usd / perf.cost_1M_online_usd.max(), color=c, alpha=0.6, edgecolor=INK)
    ax.annotate(f"{r.model}\n${r.cost_1M_online_usd:.2f} per 1M", (r.p95_ms, r.test_AUC), textcoords="offset points", xytext=(0, 24), ha="center", fontsize=9)
ax.set_xscale("log"); ax.margins(0.25)
ax.set(xlabel="p95 single-request latency (ms, log scale)", ylabel="test ROC-AUC", title="Accuracy vs latency vs cost (bubble area = cost per 1M, illustrative)")
plt.tight_layout(); plt.show()

- **Batch serving:** score the whole book overnight or hourly; cheapest per row, high latency tolerated (e.g. portfolio risk, marketing lists).
- **Online serving:** score each request in real time behind an API; pay for p95 latency and idle capacity (e.g. a loan decision at the point of application).
- Ask: is the last +0.01 AUC worth 10x latency and cost? The dollar uplift from section 4 answers that.

## 6. Offline vs online learning under drift
- Simulate 12 "weeks" of applications (1,000 per week) where the world drifts: credit score matters less, utilisation matters more.
- (a) never retrain; (b) offline: full retrain every 4 weeks on all labelled data so far; (c) online: `SGDClassifier.partial_fit` every week.
- Each week is scored **before** the model sees its labels (honest, time-ordered evaluation).

In [ ]:
from sklearn.linear_model import SGDClassifier
WEEKS, PER_WEEK = 12, 1000
weekly = [make_loans(PER_WEEK, 1000 + w, drift=w / (WEEKS - 1)) for w in range(WEEKS)]
m_static = clone(MODELS["HistGradientBoosting"]).fit(X_train, y_train)
m_offline = m_static                                                   # same start; replaced at retrain time
prep = build(None).named_steps["prep"].fit(X_train)                    # fixed preprocessing for the online model
sgd = SGDClassifier(loss="log_loss", alpha=1e-4, random_state=SEED)
Xt = prep.transform(X_train)
for _ in range(30): sgd.partial_fit(Xt, y_train, classes=[0, 1])
seen_X, seen_y = [X_train], [y_train]
auc = {"(a) never retrain": [], "(b) offline retrain / 4 wks": [], "(c) online partial_fit weekly": []}
for w, (Xw, yw) in enumerate(weekly):
    if w > 0 and w % 4 == 0:
        m_offline = clone(MODELS["HistGradientBoosting"]).fit(pd.concat(seen_X), pd.concat(seen_y))
    auc["(a) never retrain"].append(roc_auc_score(yw, m_static.predict_proba(Xw)[:, 1]))
    auc["(b) offline retrain / 4 wks"].append(roc_auc_score(yw, m_offline.predict_proba(Xw)[:, 1]))
    auc["(c) online partial_fit weekly"].append(roc_auc_score(yw, sgd.predict_proba(prep.transform(Xw))[:, 1]))
    sgd.partial_fit(prep.transform(Xw), yw)                           # labels arrive after scoring
    seen_X.append(Xw); seen_y.append(yw)

fig, ax = plt.subplots(figsize=(9, 4))
for (k, v), c in zip(auc.items(), [CORAL, TEAL, PURPLE]):
    ax.plot(range(1, WEEKS + 1), v, "-o", color=c, label=k)
for wk in (5, 9): ax.axvline(wk - 0.5, color=GOLD, ls=":", lw=1)
ax.set(xlabel="week", ylabel="AUC on that week's applications", title="Weekly AUC under drift (dotted = offline retrain)"); ax.legend(); plt.tight_layout(); plt.show()
print(pd.DataFrame({k: [np.mean(v[:4]), np.mean(v[-4:]), np.mean(v)] for k, v in auc.items()}, index=["weeks 1-4", "weeks 9-12", "mean"]).T.round(3))

- **(a) Never retrain:** zero ops cost, but quality decays silently as the world moves. Risk: nobody notices until losses appear. Needs monitoring.
- **(b) Offline retrain every N weeks:** simple, auditable, versionable (section 7); a good model class can be used. Cost: compute + validation per retrain; the model is stale between retrains.
- **(c) Online learning:** adapts every week at tiny cost, but limited to models that support `partial_fit`, and bad or late labels or a feedback loop can corrupt it quickly. Needs guardrails and rollback.
- Honest caveat: single synthetic run, 1,000 rows per week; week-to-week AUC is noisy. In production, labels (defaults) arrive months late.

## 7. Reproducibility and versioning
- Fix seeds; hash the dataset; save model + `model_card.json` per version under `artifacts/registry/`.
- v1 = best model trained on train. v2 = retrained on train + valid. Reload v1 and prove predictions are bit-identical.

In [ ]:
def dataset_hash(df): return hashlib.sha256(df.to_csv(index=False).encode()).hexdigest()[:16]

def register_model(model, version, train_df, metrics, owner, limitations, root="artifacts/registry"):
    d = pathlib.Path(root) / version; d.mkdir(parents=True, exist_ok=True)
    joblib.dump(model, d / "model.joblib")
    card = dict(model_version=version, model_type=type(model[-1] if hasattr(model, "steps") else model).__name__,
                data_hash=dataset_hash(train_df), n_train_rows=len(train_df), metrics=metrics, seed=SEED,
                library_versions=dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,
                                      sklearn=sklearn.__version__, joblib=joblib.__version__),
                date=datetime.datetime.now().isoformat(timespec="seconds"), owner=owner, limitations=limitations,
                model_sha256=hashlib.sha256((d / "model.joblib").read_bytes()).hexdigest()[:16])
    (d / "model_card.json").write_text(json.dumps(card, indent=2)); return card

LIMITS = ["Synthetic data; not for real credit decisions", "Threshold tuned to the illustrative cost matrix",
          "Drift: retrain or monitor regularly", "Not validated for fairness across regions"]
card1 = register_model(best_model, "v1", X_train, dict(test_auc=round(roc_auc_score(y_test, p_test), 4)),
                       "ml-platform-team (owner: fill in)", LIMITS)
X_tv, y_tv = pd.concat([X_train, X_valid]), pd.concat([y_train, y_valid])
model_v2 = clone(MODELS[best_name]).fit(X_tv, y_tv)
card2 = register_model(model_v2, "v2", X_tv, dict(test_auc=round(roc_auc_score(y_test, model_v2.predict_proba(X_test)[:, 1]), 4)),
                       "ml-platform-team (owner: fill in)", LIMITS + ["Retrained on train+valid"])
print(json.dumps(card1, indent=2))

In [ ]:
reloaded = joblib.load("artifacts/registry/v1/model.joblib")
same = np.array_equal(reloaded.predict_proba(X_test), best_model.predict_proba(X_test))
print("v1 reloaded predictions bit-identical to in-memory model:", same)
print("v1 vs v2 -> data_hash:", card1["data_hash"], "vs", card2["data_hash"], "| test AUC:", card1["metrics"]["test_auc"], "vs", card2["metrics"]["test_auc"])
print("Files:", sorted(str(p) for p in pathlib.Path("artifacts/registry").rglob("*") if p.is_file()))
print("Note: joblib files are only safe to reload with the same library versions (recorded in the card); never load untrusted files.")

## 8. KPI scorecard: evaluate an AI application against different KPIs (the activity)
Three candidate designs for a **customer-support assistant**:
- **A** rules + FAQ search; **B** small classifier + response templates; **C** RAG + large LLM.
- Ratings are **instructor-provided assumptions** (1 = worst, 5 = best; so a 5 on latency means fast, a 5 on cost means cheap, a 5 on hallucination risk means very low risk).

In [ ]:
RATINGS = {   # INSTRUCTOR-PROVIDED ASSUMPTIONS, not measurements
    "A: rules + FAQ search":      dict(accuracy=2, latency=5, cost=5, explainability=5, hallucination=5, maintainability=2),
    "B: small classifier + templ": dict(accuracy=3, latency=4, cost=4, explainability=3, hallucination=4, maintainability=3),
    "C: RAG + large LLM":         dict(accuracy=5, latency=2, cost=1, explainability=2, hallucination=2, maintainability=4),
}
BASE_WEIGHTS = dict(accuracy=0.38, latency=0.10, cost=0.08, explainability=0.06, hallucination=0.12, maintainability=0.26)

def score_designs(weights, ratings=RATINGS):
    s = sum(weights.values())
    return pd.Series({d: sum(weights[k] * r[k] for k in weights) / s for d, r in ratings.items()}).sort_values(ascending=False)

print("Weighted ranking (base weights):"); base = score_designs(BASE_WEIGHTS); print(base.round(2).to_string())
dbl = {**BASE_WEIGHTS, "cost": BASE_WEIGHTS["cost"] * 2}
print("\nSensitivity: cost weight doubled (weights renormalised):"); print(score_designs(dbl).round(2).to_string())
# how heavy must cost be before the winner changes?
for w in np.arange(0.08, 1.0, 0.02):
    if score_designs({**BASE_WEIGHTS, "cost": w}).index[0] != base.index[0]:
        print(f"\nWinner changes from '{base.index[0]}' to '{score_designs({**BASE_WEIGHTS, 'cost': w}).index[0]}' when the raw cost weight reaches ~{w:.2f} (base {BASE_WEIGHTS['cost']}).")
        break
else:
    print("\nWinner never changes as cost weight grows to 1.0.")

### Your turn 3: your client's context
Edit the weights for a client of your choice (e.g. a bank: explainability and hallucination risk matter most; a retailer: cost and latency).
- Which design wins? Does the ranking change vs the base case? Which rating would you challenge?

In [ ]:
my_weights = dict(BASE_WEIGHTS)   # TODO: change weights for your client, e.g. my_weights["explainability"] = 0.30
print(score_designs(my_weights).round(2).to_string())

## 9. Recap
- **Start simple:** always beat a majority-class and a rule baseline with cross-validation before adding complexity.
- **Bias-variance:** read train-vs-validation curves; fix underfit with capacity/features, overfit with data/regularisation.
- **Business metrics:** optimise the decision threshold on a cost matrix, not accuracy; report uplift in currency.
- **Trade-offs:** accuracy, latency and cost move together; inference cost dominates at scale; pick batch vs online deliberately.
- **Operate it:** plan for drift (retrain policy), version data + model + card, and score designs against weighted KPIs agreed with the client.

**OPTIONAL / homework:** re-run section 4 with `RandomForest` as the best model; add a calibration check; change the drift speed in section 6.